In [25]:
import numpy as np


def build_grm(G):
    """K = (1/m) G G^T."""
    return (G @ G.T) / G.shape[1]


def reml_nr(y, K, max_iter=100, tol=1e-6, verbose=True):
    y = np.asarray(y, dtype=float).reshape(-1, 1)
    N = y.shape[0]
    I = np.eye(N)
    Xf = np.ones((N, 1))                  # intercept only
    Ks = [K, I]

    # start at Var(y)/2 each
    v = y.var()
    theta = np.array([v / 2, v / 2])

    converged = False
    for it in range(1, max_iter + 1):
        # V, V^{-1}, P
        V = theta[0] * K + theta[1] * I
        Vi = np.linalg.inv(V)
        XtVi = Xf.T @ Vi
        P = Vi - Vi @ Xf @ np.linalg.inv(XtVi @ Xf) @ XtVi

        Py = P @ y
        PK = [P @ Ks[i] for i in range(2)]          # P K_i

        # gradient (score):  grad_i = -1/2 [ tr(P K_i) - y'P K_i P y ]
        grad = np.array([
            -0.5 * (np.trace(PK[i]) - (Py.T @ Ks[i] @ Py).item())
            for i in range(2)
        ])

        # true Hessian:  H_ij = 1/2 tr(P K_i P K_j) - y'P K_i P K_j P y
        H = np.empty((2, 2))
        for i in range(2):
            for j in range(2):
                tr_term = 0.5 * np.trace(PK[i] @ PK[j])
                quad_term = (Py.T @ Ks[i] @ P @ Ks[j] @ Py).item()
                H[i, j] = tr_term - quad_term

        # Newton-Raphson step:  theta <- theta - H^{-1} grad
        theta_new = theta - np.linalg.solve(H, grad)

        delta = np.max(np.abs(theta_new - theta))
        if verbose:
            print(f"iter {it:3d} | sg2 = {theta_new[0]:.5f} | "
                  f"se2 = {theta_new[1]:.5f} | dmax = {delta:.2e}")
        theta = theta_new
        if delta < tol:
            converged = True
            break

    sg, se = theta
    return {"sigma_g2": sg, "sigma_e2": se, "h2": sg / (sg + se),
            "n_iter": it, "converged": converged}


if __name__ == "__main__":       
    s2gxg = .5
    s2e = 1 - s2gxg
    n, m = 1000, 1000

    X = rng.standard_normal((n, m))
    X = (X - X.mean(0)) / X.std(0)          
    beta = rng.standard_normal((m, 1)) * np.sqrt(s2gxg / m)
    e    = rng.standard_normal((n, 1)) * np.sqrt(s2e)
    y    = X @ beta + e
    y -= np.mean(y)

    K = build_grm(X)                      

    print(f"True: sg2={s2gxg}, se2={s2e}, h2={s2gxg/(s2gxg+s2e):.4f}\n")
    out = reml_nr(y, K)
    print(f"\nEstimated: sg2={out['sigma_g2']:.4f}, se2={out['sigma_e2']:.4f}, "
          f"h2={out['h2']:.4f}  ({out['n_iter']} iters, conv={out['converged']})")

True: sg2=0.5, se2=0.5, h2=0.5000

iter   1 | sg2 = 0.50269 | se2 = 0.50167 | dmax = 7.35e-03
iter   2 | sg2 = 0.50285 | se2 = 0.50187 | dmax = 2.05e-04
iter   3 | sg2 = 0.50285 | se2 = 0.50187 | dmax = 1.67e-07

Estimated: sg2=0.5028, se2=0.5019, h2=0.5005  (3 iters, conv=True)
